# Deployment Results — Plotting Notebook

## 1 — Configuration

In [ ]:
# Path to saved deployment JSON file 
JSON_PATH = 'runs_deployment/3rd_Run_1000_episodes/designs/deployment_designs.json'   # update this path as needed

# Output file prefix 
PREFIX = 'deployment'

# Environment constraint overrides 
ENV_PARAMS = {}

SHOW_INLINE = True

## 2 — Load data

In [ ]:
import matplotlib
if SHOW_INLINE:
    matplotlib.use('module://matplotlib_inline.backend_inline')
else:
    matplotlib.use('Agg')

import matplotlib.pyplot as plt
from DeployPlots import DeploymentPlotter

plotter = DeploymentPlotter(JSON_PATH, env_params=ENV_PARAMS)

print(f"\nBest episode : #{plotter.best_ep['episode']}")
print(f"Reason       : {plotter.best_label}")
d = plotter.best_ep['final']
print(f"VNin  = {d['vn_in_nV']:.4f} nV/√Hz")
print(f"Fc    = {d['fc_Hz']/1e3:.2f} kHz")
print(f"Ibias = {d['ibias_uA']:.2f} µA")
print(f"Area  = {d['area_um2']:.1f} µm²")
print(f"Success: {'✓' if d['success'] else '✗'}")

## 3 — Generate all figures

In [ ]:
figs = plotter.plot_all(prefix=PREFIX, show=SHOW_INLINE)

## 4 — Episode summary table

In [ ]:
import json

print(f"{'Ep':>4}  {'VNin':>10}  {'Fc(kHz)':>8}  {'Ibias(µA)':>10}  {'Area(µm²)':>10}  {'Steps':>6}  {'OK?':>4}")
print('-' * 65)
for ep in plotter.episode_records:
    d = ep['final']
    vnin_ok = d['vn_in_nV'] < plotter.env['vnin_target']
    fc_ok   = d['fc_ok']
    area_ok = d['area_um2'] <= plotter.env['area_budget_um2']
    all_ok  = vnin_ok and fc_ok and area_ok
    tag     = '✓' if all_ok else ('~' if (vnin_ok and fc_ok) else '✗')
    best_marker = '  ← BEST' if ep['episode'] == plotter.best_ep['episode'] else ''
    print(
        f"{ep['episode']:>4}  "
        f"{d['vn_in_nV']:>10.4f}  "
        f"{d['fc_Hz']/1e3:>8.2f}  "
        f"{d['ibias_uA']:>10.2f}  "
        f"{d['area_um2']:>10.1f}  "
        f"{ep['steps']:>6}  "
        f"{tag:>4}{best_marker}"
    )
print('\n  ✓ = all constraints met   ~ = VNin+Fc met (area over)   ✗ = failed')

## 5 — Inspect a specific episode

Chose episode to plot trajectory.

In [ ]:
# Select episode 
EPISODE_NUMBER = 1    # change this to any episode you want to inspect

fig = plotter.plot_episode(EPISODE_NUMBER, prefix=PREFIX, show=SHOW_INLINE)
print(f"\nEpisode {EPISODE_NUMBER} plot saved.")

## 6 — Browse all episodes

Loop over a range of episodes and save each one.

In [ ]:
# Change the range to whichever episodes you want 
EPISODES_TO_PLOT = range(1, len(plotter.episode_records) + 1)   # all episodes
# EPISODES_TO_PLOT = [1, 5, 10, 25, 50]                         # or a specific list

for ep_num in EPISODES_TO_PLOT:
    plotter.plot_episode(ep_num, prefix=PREFIX, show=False)

print("\nAll selected episodes saved.")